Fintech Loan Risk Project Data Cleaning Pipeline

In [1]:
import pandas as pd
import numpy as np

Load dataset

In [3]:
df = pd.read_csv("loan_data.csv", low_memory=False)
print("Initial Shape:", df.shape)

Initial Shape: (466285, 75)


Basic Data Inspection

In [4]:
print("\nColumns:")
print(df.columns)

print("\nMissing Values:")
print(df.isnull().sum().sort_values(ascending=False).head(20))


Columns:
Index(['Unnamed: 0', 'id', 'member_id', 'loan_amnt', 'funded_amnt',
       'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade',
       'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc',
       'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url',
       'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti',
       'delinq_2yrs', 'earliest_cr_line', 'inq_last_6mths',
       'mths_since_last_delinq', 'mths_since_last_record', 'open_acc',
       'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
       'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt',
       'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
       'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
       'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d',
       'collections_12_mths_ex_med', 'mths_since_last_major_derog',
       'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint',
       'v

Remove Columns With Too Many Missing Values

In [9]:
missing_threshold = 0.5
missing_ratio = df.isnull().mean()

cols_to_drop = missing_ratio[missing_ratio > missing_threshold].index
df.drop(columns=cols_to_drop, inplace=True)

print("\nDropped columns with >50% missing values")
df.shape


Dropped columns with >50% missing values


(466285, 54)

In [12]:
df.head()

,Unnamed: 0,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,...,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,policy_code,application_type,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim
0,0,1077501,1296599,5000,5000,4975.0,36 months,10.65,162.87,B,...,171.62,NaN,Jan-16,0.0,1,INDIVIDUAL,0.0,NaN,NaN,NaN
1,1,1077430,1314167,2500,2500,2500.0,60 months,15.27,59.83,C,...,119.66,NaN,Sep-13,0.0,1,INDIVIDUAL,0.0,NaN,NaN,NaN
2,2,1077175,1313524,2400,2400,2400.0,36 months,15.96,84.33,C,...,649.91,NaN,Jan-16,0.0,1,INDIVIDUAL,0.0,NaN,NaN,NaN
3,3,1076863,1277178,10000,10000,10000.0,36 months,13.49,339.31,C,...,357.48,NaN,Jan-15,0.0,1,INDIVIDUAL,0.0,NaN,NaN,NaN
4,4,1075358,1311748,3000,3000,3000.0,60 months,12.69,67.79,B,...,67.79,Feb-16,Jan-16,0.0,1,INDIVIDUAL,0.0,NaN,NaN,NaN


Important columns for loan risk model

In [13]:
important_columns = [
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "purpose",
    "dti",
    "delinq_2yrs",
    "revol_util",
    "total_acc",
    "loan_status"
]

df = df[important_columns]

print("Shape after column selection:", df.shape)

Shape after column selection: (466285, 15)


Handling missing values 

In [19]:
numeric_cols = df.select_dtypes(include=["float64", "int64"]).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

categorical_cols = df.select_dtypes(include=["object"]).columns

for col in catergorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

C:\Users\navya\AppData\Local\Temp\ipykernel_20568\2302546054.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].fillna(df[col].median())
C:\Users\navya\AppData\Local\Temp\ipykernel_20568\2302546054.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].fillna(df[col].median())
C:\Users\navya\AppData\Local\Temp\ipykernel_20568\2302546054.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] =

In [20]:
numeric_cols

Index(['loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti',
       'delinq_2yrs', 'revol_util', 'total_acc'],
      dtype='object')

In [21]:
categorical_cols

Index(['term', 'grade', 'emp_length', 'home_ownership', 'verification_status',
       'purpose', 'loan_status'],
      dtype='object')

Clean Employment Length

In [22]:
df["emp_length"] = df["emp_length"].str.extract("(\d+)")
df["emp_length"] = df["emp_length"].astype(float)
df["emp_length"] = df["emp_length"].fillna(0)

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\navya\AppData\Local\Temp\ipykernel_20568\1950967753.py:1: SyntaxWarning: invalid escape sequence '\d'
  df["emp_length"] = df["emp_length"].str.extract("(\d+)")


In [23]:
df["emp_length"]

0         10.0
1          1.0
2         10.0
3         10.0
4          1.0
          ... 
466280     4.0
466281    10.0
466282     7.0
466283     3.0
466284    10.0
Name: emp_length, Length: 466285, dtype: float64

Convert Interest Rate

In [26]:
df["int_rate"] = df["int_rate"].astype(float)
df["int_rate"]

0         10.65
1         15.27
2         15.96
3         13.49
4         12.69
          ...  
466280    14.47
466281    19.97
466282    16.99
466283     7.90
466284    19.20
Name: int_rate, Length: 466285, dtype: float64

Convert Revolving Utilization

In [27]:
df["revol_util"].dtype

dtype('float64')

In [29]:
df["revol_util"] = pd.to_numeric(df["revol_util"], errors="coerce")
df["revol_util"] = df["revol_util"].fillna(df["revol_util"].median())

In [30]:
df["revol_util"]

0         83.7
1          9.4
2         98.5
3         21.0
4         53.9
          ... 
466280    77.6
466281    46.3
466282    51.1
466283    21.5
466284    70.8
Name: revol_util, Length: 466285, dtype: float64

Convert Loan Term

In [31]:
df["term"] = df["term"].str.extract("(\d+)")
df["term"] = df["term"].astype(int)
df["term"]

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\navya\AppData\Local\Temp\ipykernel_20568\3857821538.py:1: SyntaxWarning: invalid escape sequence '\d'
  df["term"] = df["term"].str.extract("(\d+)")


0         36
1         60
2         36
3         36
4         60
          ..
466280    60
466281    60
466282    60
466283    36
466284    36
Name: term, Length: 466285, dtype: int64

Target Variable Cleaning

In [32]:
df = df[df["loan_status"].isin(["Fully Paid", "Charged Off"])]

df["loan_status"] = df["loan_status"].map({
    "Fully Paid": 0,
    "Charged Off": 1
})
df["loan_status"]

0         0
1         1
2         0
3         0
5         0
         ..
466276    1
466277    1
466278    0
466281    1
466283    0
Name: loan_status, Length: 227214, dtype: int64

Feature Engineering (Fintech Risk)

In [33]:
df["loan_income_ratio"] = df["loan_amnt"] / df["annual_inc"]
df["credit_risk_score"] = df["dti"] * df["loan_income_ratio"]
df["payment_burden"] = df["installment"] / df["annual_inc"]

Remove Outliers

In [35]:
df = df[df["annual_inc"] < df["annual_inc"].quantile(0.99)]
df = df[df["loan_amnt"] < df["loan_amnt"].quantile(0.99)]

Encode Categorical Variables

In [47]:
df = pd.get_dummies(df, columns=[
    "grade",
    "home_ownership",
    "verification_status",
    "purpose"
], drop_first=True). #to avoid dummy variable trap

SyntaxError: invalid syntax (2521980991.py, line 6)

In [41]:
df.head()

,loan_amnt,term,int_rate,installment,emp_length,annual_inc,dti,delinq_2yrs,revol_util,total_acc,...,purpose_home_improvement,purpose_house,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding
0,5000,36,10.65,162.87,10.0,24000.0,27.65,0.0,83.7,9.0,...,False,False,False,False,False,False,False,False,False,False
1,2500,60,15.27,59.83,1.0,30000.0,1.00,0.0,9.4,4.0,...,False,False,False,False,False,False,False,False,False,False
2,2400,36,15.96,84.33,10.0,12252.0,8.72,0.0,98.5,10.0,...,False,False,False,False,False,False,False,True,False,False
3,10000,36,13.49,339.31,10.0,49200.0,20.00,0.0,21.0,37.0,...,False,False,False,False,False,True,False,False,False,False
5,5000,36,7.90,156.46,3.0,36000.0,11.20,0.0,28.3,12.0,...,False,False,False,False,False,False,False,False,False,True


In [48]:
df.columns 

Index(['loan_amnt', 'term', 'int_rate', 'installment', 'emp_length',
       'annual_inc', 'dti', 'delinq_2yrs', 'revol_util', 'total_acc',
       'loan_status', 'loan_income_ratio', 'credit_risk_score',
       'payment_burden', 'grade_B', 'grade_C', 'grade_D', 'grade_E', 'grade_F',
       'grade_G', 'home_ownership_MORTGAGE', 'home_ownership_NONE',
       'home_ownership_OTHER', 'home_ownership_OWN', 'home_ownership_RENT',
       'verification_status_Source Verified', 'verification_status_Verified',
       'purpose_credit_card', 'purpose_debt_consolidation',
       'purpose_educational', 'purpose_home_improvement', 'purpose_house',
       'purpose_major_purchase', 'purpose_medical', 'purpose_moving',
       'purpose_other', 'purpose_renewable_energy', 'purpose_small_business',
       'purpose_vacation', 'purpose_wedding'],
      dtype='object')

Remove Duplicates

In [50]:
df.drop_duplicates(inplace=True)

FINAL CHECK

In [52]:
print("\nFinal Dataset Shape:", df.shape)
print("\nRemaining Missing Values:")
print(df.isnull().sum().sum())


Final Dataset Shape: (218832, 40)

Remaining Missing Values:
0


Save Clean Dataset

In [55]:
df.to_csv("cleaned_loan_data.csv", index=False)

print("\nCleaned dataset saved successfully!")


Cleaned dataset saved successfully!
